# Urban and rural surrounding areas selection

***
> This work is licensed under a [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0)
>
> ![Creative Commons License >](https://i.creativecommons.org/l/by/4.0/88x31.png)

Adapted from the URCLIMASK example notebook (https://github.com/FPS-URB-RCC/URCLIMASK, CC BY 4.0).

**Role in the paper.** Produces Fig. 1: the seven evaluation methods illustrated for Rennes (urban/rural masks, selected grid cells and stations). It repeats the mask and method set-up of `Eurocities_urclimmask_Rennes_.ipynb` and reads the p2a plot data saved by `method_2_blended_approach/UHI_blended_approach_Rennes.ipynb`. The final figure is the `_v2` version saved by the last cells.

| Code name (columns / variables) | Paper name |
|---|---|
| `p2p_un_ucdb` | p2p_UCN_uc |
| `p2p_un_lcz` | p2p_UCN_lcz |
| `p2p_ms_ucdb` | p2p_MS_uc |
| `urclim_un_ucdb` | a2a_UCN_uc |
| `urclim_un_lcz` | a2a_UCN_lcz |
| `urclim_ms_ucdb` | a2a_MS_uc |
| `blend_ms_ucdb` (method 2, blended) | p2a_MS_dist |

## Load libraries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import cf_xarray as cfxr
from pathlib import Path
import metobs_toolkit

from rasterio import warp

import cftime
import pandas as pd
import matplotlib as plt
from pprint import pprint
import importlib
import cartopy.crs as ccrs
import geopandas as gpd
import rasterio
import sys
import pickle

from urclimask.urban_areas import (
    UrbanVicinity
)
from urclimask.utils import (
    kelvin2degC,
    traverseDir,
    fix_360_longitudes,
    RCM_DICT,
    load_ucdb_city,
    align_to_reference_grid,
    classify_stations_from_ucdb_city,
    plot_UHI_p2p_cycle,
    compute_UHI_p2p, 
    from_latlon_to_xy,
    classify_stations_from_LCZ, 
    plot_polygon_boundaries, 
    build_polygons_from_urmask, 
    plot_blend_on_ax
)

from urclimask.UHI_analysis import (
    UrbanIsland, 
    
)
from urclimask.GHCNd_stations import (
    load_ghcnd_stations,
    get_ghcnd_df,
    get_valid_timeseries, 
    available_vars,
    inside_city
)
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# Urclimask

## Urclimask settings

In [ ]:
city = 'Rennes'
lon_city =  -1.64
lat_city = 48.125

start = "2018-01-01"
stop = "2018-12-31"

network_og = network = 'city_network'
subset_og = subset = '_gapfree'

#Ghent (51.05454277446639, 3.7249005076743438)
domain = 'EUROCITIES'
domain_resolution = 2.5

model = 'ALAROSFX'
scenario = "evaluation" 

urban_var = 'sfturf'

urban_th = 40
urban_sur_th = 10
orog_diff = 100
sftlf_th = 60
ratio_r2u = 2
min_city_size = 5
lon_lim = 0.13
lat_lim = 0.13

Create a directory to save the results for this particular city and model

In [ ]:
ROOT = Path.cwd().parent.parent.parent
data_dir =  ROOT / f'data/'
plot_dir =  ROOT / f'results/figures/{city}/method_2_urclimask_{network}'
os.makedirs(plot_dir, exist_ok=True)
save_masks_dir = ROOT / f'data/metadata/masks/'
os.makedirs(save_masks_dir, exist_ok=True)
save_dir =  ROOT / f'results/data/{city}/'
os.makedirs(save_dir, exist_ok=True)
base_filename = f'{city}-{domain}_ECMWF-ERA5_{scenario}_r1i1p1f1_{network}_{subset}'

blended_approach_pkl = ROOT / f'results/data/{city}/method_2_blended_approach/{city}_p2a_plot_data.pkl'

## Define urban areas and vicinity

### Load city shapefile and static variables
Load static variables, such as urban fraction (sfturf), terrain elevation (orography) or land-sea fraction (sftlf).

In [ ]:
ucdb_city = load_ucdb_city(f'{data_dir}/metadata/UCDB_shapefiles', city)

static_input_path = f'{data_dir}/metadata/simulations/static_variables_eurocities.nc'
static_input = xr.open_dataset(static_input_path,engine='netcdf4')
static_input = fix_360_longitudes(static_input)

ds_sfturf, ds_orog, ds_sftlf = static_input, static_input, static_input


In [ ]:
#Add parameters to the function
URBAN = UrbanVicinity(
    urban_sur_th = urban_sur_th,
    orog_diff = orog_diff,
    sftlf_th = sftlf_th,
    ratio_r2u = ratio_r2u,
    min_city_size = min_city_size,
    lon_city = lon_city,
    lat_city = lat_city,
    lon_lim = lon_lim,
    lat_lim = lat_lim,
    model = model,
    domain = domain,
    urban_th = urban_th,
    urban_var = urban_var
)

#crop around the city
ds_sfturf = URBAN.crop_area_city(ds = ds_sfturf, res = domain_resolution)
ds_orog = URBAN.crop_area_city(ds = ds_orog, res = domain_resolution)
ds_sftlf = URBAN.crop_area_city(ds = ds_sftlf, res = domain_resolution)

#Define masks using the parameters above
sfturf_mask, sfturf_sur_mask, orog_mask, sftlf_mask = URBAN.define_masks(
    ds_sfturf = ds_sfturf, 
    ds_orog = ds_orog, 
    ds_sftlf = ds_sftlf,
)

### Define rural vicinity areas

We use an iterative algorithm to select the urban vicinity excluding masked areas

In [ ]:
urmask = URBAN.select_urban_vicinity(
    sfturf_mask = sfturf_mask, 
    orog_mask = orog_mask,
    sftlf_mask = sftlf_mask,
    sfturf_sur_mask = sfturf_sur_mask,
    debug = True,
    max_iterations=200,
)

We can plot the masks and the urban vs vicinity areas. The latter are are passed via the `urban_areas` parameter. They are hidden if set to `None`.

In [ ]:
fig = URBAN.plot_static_variables(ds_sfturf = ds_sfturf, 
                                  ds_orog = ds_orog, 
                                  ds_sftlf = ds_sftlf,
                                  sfturf_mask = sfturf_mask, 
                                  orog_mask = orog_mask, 
                                  sftlf_mask = sftlf_mask,
                                  urban_areas = urmask,
                                  ucdb_city = ucdb_city,
                                 composite = False) 
fig.savefig(f"{plot_dir}/urmask_{base_filename}_fx_.pdf", bbox_inches='tight')

In [ ]:
fig = URBAN.plot_static_variables(ds_sfturf = ds_sfturf, 
                                  ds_orog = ds_orog, 
                                  ds_sftlf = ds_sftlf,
                                  sfturf_mask = sfturf_mask, 
                                  orog_mask = orog_mask, 
                                  sftlf_mask = sftlf_mask,
                                  urban_areas = urmask,
                                  ucdb_city = ucdb_city,
                                 composite = True, )
fig.savefig(f"{plot_dir}/urmask_{base_filename}_fx.pdf", bbox_inches='tight')

Save these masks and urban fraction to NetCDF

In [ ]:

urmask.to_netcdf(f"{save_masks_dir}/urmask_{base_filename}_fx.nc")

## Load simulation data

In [ ]:
var='tasmin'


In [ ]:
path_monmean = f"{data_dir}/processed/simulations/monthly_means/{domain}_operational_cy43_{var}_monmean_*.nc"
da_var = xr.open_mfdataset(sorted(glob.glob(path_monmean)), engine="netcdf4", chunks="auto").rename({'tas':var}).sel(time=slice(start,stop)) 

ds_RCM_urb = URBAN.crop_area_city(ds = da_var, res = domain_resolution)
ds_RCM_urb["time"] = ds_RCM_urb.indexes["time"].to_period("M").to_timestamp()
ds_RCM_urb

In [ ]:
# Align static information to model data coordinates (in case their is a rounding error in the grid coordinates causing a mismatch in the grid)
urmask, alignment_status = align_to_reference_grid(urmask, ds_RCM_urb, coord_names=['x', 'y'])
print(alignment_status)

## Load Observational data

### Load urban climate network data

In [ ]:
start_o = start #because 2020, the year represented by the Urban network is not in the meteostations network
stop_o = stop

# load station observation data (preproccessed: monthly mean of daily data)
obs_path = f"{data_dir}/processed/observations/{city}/{network}/datasets/{subset}/{city}_{network}{subset}_monthlymean_daily{var[3:]}.nc"
ds_obs = xr.open_dataset(obs_path, engine="netcdf4", chunks="auto").sel(time=slice(start_o,stop_o))
obs_timeseries = ds_obs.to_dataframe().copy()

stations = obs_timeseries.columns.to_list()
print(f"{len(stations)} available stations: {stations}")

Prepare obs_attributes metadata for two urban/rural classfiction approaches: core based (ucdb polygon) and LCZ based

In [ ]:
pd_meta = pd.read_csv(f"{data_dir}/processed/observations/{city}/{network}/{city}_{network}_metadata.csv")
pd_meta.rename(columns={'name':'code'}, inplace=True)
obs_attributes = classify_stations_from_ucdb_city(pd_meta, ucdb_city)
obs_attributes = obs_attributes[obs_attributes["code"].isin(stations)]

pd_meta = pd.read_csv(f"{data_dir}/processed/observations/{city}/{network}/{city}_{network}_metadata.csv")
pd_meta.rename(columns={'name':'code'}, inplace=True)
obs_attributes_lczbased = classify_stations_from_LCZ(pd_meta)
obs_attributes_lczbased = obs_attributes_lczbased[obs_attributes_lczbased["code"].isin(stations)]
obs_attributes_lczbased

In [ ]:
for station in obs_timeseries.columns:
    print(f"Variable: {station}, nan count: {obs_timeseries[station].isnull().sum().item()}")

### Load synoptic stations (national meteorological stations)

In [ ]:
network_m='meteo_stations'
subset_m= ''
start_m = start
stop_m = stop


# load station observation data (preproccessed: monthly mean of daily data)
obs_path_tasmin_m = f"{data_dir}/processed/observations/{city}/{network_m}/datasets/{subset_m}/{city}_{network_m}{subset_m}_monthlymean_daily{var[3:]}.nc"
ds_obs_tasmin_m = xr.open_dataset(obs_path_tasmin_m, engine="netcdf4", chunks="auto").sel(time=slice(start,stop))
obs_timeseries_m = ds_obs_tasmin_m.to_dataframe().copy()
#obs_timeseries.drop('UKE00105909', axis='columns')
stations_m = obs_timeseries_m.columns.to_list()
print(f"{len(stations_m)} available stations: {stations_m}")
pd_meta = pd.read_csv(f"{data_dir}/processed/observations/{city}/{network_m}/{city}_{network_m}_metadata.csv")
pd_meta.rename(columns={'name':'code'}, inplace=True)
obs_attributes_m = classify_stations_from_ucdb_city(pd_meta, ucdb_city)
obs_attributes_m = obs_attributes_m[obs_attributes_m["code"].isin(stations_m)]
obs_attributes_m

In [ ]:

for station in obs_timeseries_m.columns:
    print(f"station: {station}, nan count: {obs_timeseries_m[station].isnull().sum().item()}")


## UHI Urclim analysis

In [ ]:
proj_dict = ds_RCM_urb.crs.attrs
plot_proj = ccrs.LambertConformal(central_latitude=proj_dict["latitude_of_projection_origin"], central_longitude=proj_dict["longitude_of_central_meridian"], standard_parallels=proj_dict["standard_parallel"], globe=ccrs.Globe(semimajor_axis=proj_dict["earth_radius"], semiminor_axis=proj_dict["earth_radius"]))
plot_proj

obs_attributes[["x", "y"]] = obs_attributes.apply( lambda row: from_latlon_to_xy(row["lat"], row["lon"], plot_proj), axis=1, result_type="expand")
obs_attributes_m[["x", "y"]] = obs_attributes_m.apply( lambda row: from_latlon_to_xy(row["lat"], row["lon"], plot_proj), axis=1, result_type="expand")
obs_attributes_lczbased[["x", "y"]]= obs_attributes_lczbased.apply( lambda row: from_latlon_to_xy(row["lat"], row["lon"], plot_proj), axis=1, result_type="expand")

for obs_attr, method in [(obs_attributes, 'UCN_uc'), (obs_attributes_lczbased, 'UCN_lcz'), (obs_attributes_m, 'MS_uc')]:
    p2p_mask = xr.full_like(urmask["urmask"], np.nan, dtype=float)
    # Loop over stations
    for _, station in obs_attr.iterrows():

        # Find nearest grid point
        ix = np.abs(urmask.x.values - station["x"]).argmin()
        iy = np.abs(urmask.y.values - station["y"]).argmin()

        # Assign inside_city value (0 or 1)
        p2p_mask.values[iy, ix] = station["inside_city"]
    urmask[f"p2p_{method}"] = p2p_mask
urmask

### UHI Urclim : urban network , ubcd classification

In [ ]:
UHI = UrbanIsland(
    ds=ds_RCM_urb[var],
    urban_vicinity=urmask,
    anomaly="abs",  # or "rel"
    obs_attributes=obs_attributes,
    obs_timeseries=obs_timeseries
)
report = UHI.check_obs_sim_overlap()

#calculate the annual cycle of the UHI for thobservations
UHI.compute_annual_cycle()
ds_anomaly = UHI.ds_annual_cycle

rural_anomaly = ds_anomaly.where(UHI.urban_vicinity['urmask'] == 0)
urban_anomaly = ds_anomaly.where(UHI.urban_vicinity['urmask'] == 1)

y_dim, x_dim = UHI._get_spatial_dim_names(ds_anomaly)       

urban_mean = urban_anomaly.mean(dim = [y_dim, x_dim]).compute()
rural_mean = rural_anomaly.mean(dim = [y_dim, x_dim]).compute()

uhi_urclim_un_ucdb_sim = urban_mean.values
uhi_urclim_un_ucdb_obs = UHI.obs_annual_cycle['urban_mean'].values


fig = UHI.plot_UI_map(
    city_name = city,
    ucdb_city = ucdb_city, 
)
fig.savefig(f"{plot_dir}/anamolymap_UN_UCDBclass_{var}.pdf", bbox_inches='tight')

fig = UHI.plot_UI_annual_cycle(
    ds_anomaly = ds_anomaly,
    obs_anomaly = UHI.obs_annual_cycle,
    percentiles = [5,25],
    gridcell_series = True, 
    city_name = city,
    vmin = -1, vmax =3.5)
fig.savefig(f"{plot_dir}/annualcycle_UN_UCDBclass_{var}.pdf", bbox_inches='tight')

### UHI Urclim: urban network, LCZ classification

In [ ]:
UHI_lcz = UrbanIsland(
    ds=ds_RCM_urb[var],
    urban_vicinity=urmask,
    anomaly="abs",  # or "rel"
    obs_attributes=obs_attributes_lczbased,
    obs_timeseries=obs_timeseries
)
report = UHI_lcz.check_obs_sim_overlap()

#calculate the annual cycle of the UHI for thobservations
UHI_lcz.compute_annual_cycle()
ds_anomaly = UHI_lcz.ds_annual_cycle

rural_anomaly = ds_anomaly.where(UHI_lcz.urban_vicinity['urmask'] == 0)
urban_anomaly = ds_anomaly.where(UHI_lcz.urban_vicinity['urmask'] == 1)

y_dim, x_dim = UHI_lcz._get_spatial_dim_names(ds_anomaly)       

urban_mean = urban_anomaly.mean(dim = [y_dim, x_dim]).compute()
rural_mean = rural_anomaly.mean(dim = [y_dim, x_dim]).compute()

uhi_urclim_un_lcz_sim = urban_mean.values
uhi_urclim_un_lcz_obs = UHI_lcz.obs_annual_cycle['urban_mean'].values


fig = UHI_lcz.plot_UI_map(
    city_name = city,
    ucdb_city = ucdb_city, 
)
fig.savefig(f"{plot_dir}/anamolymap_UN_LCZclass_{var}", bbox_inches='tight')

fig = UHI_lcz.plot_UI_annual_cycle(
    ds_anomaly = ds_anomaly,
    obs_anomaly = UHI_lcz.obs_annual_cycle,
    percentiles = [5,25],
    gridcell_series = True, 
    city_name = city,
    vmin = -1, vmax =3.5)
fig.savefig(f"{plot_dir}/annualcycle_UN_LCZclass_{var}.pdf", bbox_inches='tight')

### UHI urclim, national meteorological stations, UBCD classificatino

In [ ]:
UHI_m = UrbanIsland(
    ds=ds_RCM_urb[var],
    urban_vicinity=urmask,
    anomaly="abs",  # or "rel"
    obs_attributes=obs_attributes_m,
    obs_timeseries=obs_timeseries_m
)
report = UHI_m.check_obs_sim_overlap()

#calculate the annual cycle of the UHI for thobservations
UHI_m.compute_annual_cycle()
ds_anomaly = UHI_m.ds_annual_cycle

rural_anomaly = ds_anomaly.where(UHI_m.urban_vicinity['urmask'] == 0)
urban_anomaly = ds_anomaly.where(UHI_m.urban_vicinity['urmask'] == 1)

y_dim, x_dim = UHI_m._get_spatial_dim_names(ds_anomaly)       

urban_mean = urban_anomaly.mean(dim = [y_dim, x_dim]).compute()
rural_mean = rural_anomaly.mean(dim = [y_dim, x_dim]).compute()

uhi_urclim_ms_sim = urban_mean.values
uhi_urclim_ms_obs = UHI_m.obs_annual_cycle['urban_mean'].values


fig = UHI_m.plot_UI_map(
    city_name = city,
    ucdb_city = ucdb_city, 
)
fig.savefig(f"{plot_dir}/anamolymap_MS_UCDBclass_{var}", bbox_inches='tight')

fig = UHI_m.plot_UI_annual_cycle(
    ds_anomaly = ds_anomaly,
    obs_anomaly = UHI_m.obs_annual_cycle,
    percentiles = [5,25],
    gridcell_series = True, 
    city_name = city,
    vmin = -1, vmax =3.5)

fig.savefig(f"{plot_dir}/annualcycle_MS_UCDBclass_{var}.pdf", bbox_inches='tight')

# Point to point analysis 

In [ ]:
proj_dict = ds_RCM_urb.crs.attrs
plot_proj = ccrs.LambertConformal(central_latitude=proj_dict["latitude_of_projection_origin"], central_longitude=proj_dict["longitude_of_central_meridian"], standard_parallels=proj_dict["standard_parallel"], globe=ccrs.Globe(semimajor_axis=proj_dict["earth_radius"], semiminor_axis=proj_dict["earth_radius"]))
plot_proj

In [ ]:
proj_dict = ds_RCM_urb.crs.attrs
plot_proj = ccrs.LambertConformal(central_latitude=proj_dict["latitude_of_projection_origin"], central_longitude=proj_dict["longitude_of_central_meridian"], standard_parallels=proj_dict["standard_parallel"], globe=ccrs.Globe(semimajor_axis=proj_dict["earth_radius"], semiminor_axis=proj_dict["earth_radius"]))
plot_proj

obs_attributes[["x", "y"]] = obs_attributes.apply( lambda row: from_latlon_to_xy(row["lat"], row["lon"], plot_proj), axis=1, result_type="expand")
obs_attributes_m[["x", "y"]] = obs_attributes_m.apply( lambda row: from_latlon_to_xy(row["lat"], row["lon"], plot_proj), axis=1, result_type="expand")
obs_attributes_lczbased[["x", "y"]]= obs_attributes_lczbased.apply( lambda row: from_latlon_to_xy(row["lat"], row["lon"], plot_proj), axis=1, result_type="expand")

for obs_attr, method in [(obs_attributes, 'UCN_uc'), (obs_attributes_lczbased, 'UCN_lcz'), (obs_attributes_m, 'MS_uc')]:
    p2p_mask = xr.full_like(urmask["urmask"], np.nan, dtype=float)
    # Loop over stations
    for _, station in obs_attr.iterrows():

        # Find nearest grid point
        ix = np.abs(urmask.x.values - station["x"]).argmin()
        iy = np.abs(urmask.y.values - station["y"]).argmin()

        # Assign inside_city value (0 or 1)
        p2p_mask.values[iy, ix] = station["inside_city"]
    urmask[f"p2p_{method}"] = p2p_mask
urmask

In [ ]:
fig,ax = plt.subplots(figsize=(10, 6))
uhi_p2p_un_core= compute_UHI_p2p( obs_attributes, obs_timeseries, ds_RCM_urb, proj=plot_proj,mean=False, variable=var)
uhi_p2p_un_ucdb_obs, uhi_p2p_un_ucdb_sim = plot_UHI_p2p_cycle( ax, uhi_p2p_un_core, mean=False,  percentiles=[20], plot_individual=False)
plt.show()

In [ ]:
fig,ax = plt.subplots(figsize=(10, 6))
uhi_p2p_un_lcz = compute_UHI_p2p( obs_attributes_lczbased, obs_timeseries, ds_RCM_urb, proj=plot_proj,mean=False, variable=var)
uhi_p2p_un_lcz_obs, uhi_p2p_un_lcz_sim = plot_UHI_p2p_cycle( ax, uhi_p2p_un_lcz, mean=False,  percentiles=[20], plot_individual=False)
plt.show()

In [ ]:
uhi_un_p2p_fig,ax = plt.subplots(figsize=(10, 6))
uhi_p2p_ms = compute_UHI_p2p( obs_attributes_m, obs_timeseries_m, ds_RCM_urb, proj=plot_proj,mean=False, variable=var)
uhi_p2p_ms_obs, uhi_p2p_ms_sim = plot_UHI_p2p_cycle( ax, uhi_p2p_ms, mean=False,  percentiles=[20], plot_individual=False)
plt.show()

# Plot methods on map 

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': plot_proj})
ax = UHI_lcz.plot_UI_urclim_visualisation(proj = plot_proj, ax=ax, mask_name = 'p2p_UCN_lcz', ucdb_city = ucdb_city)
plt.show()

In [ ]:
sim_ts = xr.open_dataset(f"{save_dir}/method_2_blended_approach\{city}_UHI_method2_simulation.nc", engine="netcdf4")
obs_ts = xr.open_dataset(f"{save_dir}/method_2_blended_approach\{city}_UHI_method2_observations.nc", engine="netcdf4")

uhi_blend_obs = obs_ts[f'UHI_{var}'].groupby('time.month').mean().values
uhi_blend_sim = sim_ts[f'UHI_{var}'].groupby('time.month').mean().values

In [ ]:

with open(blended_approach_pkl, "rb") as f:
    plot_data = pickle.load(f)

ds_crp_blend = plot_data["da_crp"]
x = ds_crp_blend.x
y = ds_crp_blend.y

rural_polygon_blend = plot_data["rural_polygon"]

x_station_blend = plot_data["x_station"]
y_station_blend = plot_data["y_station"]

x_sim_blend = plot_data["x_sim"]
y_sim_blend = plot_data["y_sim"]

city_blend = plot_data["city"]

da_anom = ds_crp_blend['anom_tasmin'].mean(dim='time')
urban_mask_blend = xr.DataArray(
    np.full(da_anom.shape, np.nan, dtype=float),
    coords=da_anom.coords,
    dims=da_anom.dims,
    name="urban_mask_blend",
)

ix = np.abs(urban_mask_blend.x.values - x_sim_blend).argmin()
iy = np.abs(urban_mask_blend.y.values - y_sim_blend).argmin()

print("ix, iy:", ix, iy)
print("x, y:", urban_mask_blend.x.values[ix], urban_mask_blend.y.values[iy])

urban_mask_blend.values[iy, ix] = 1

print("Value:", urban_mask_blend.values[iy, ix])
ds_urban_mask_blend = urban_mask_blend.to_dataset(name="urban_mask_blend")
urban_polygon_blend = build_polygons_from_urmask(ds_urban_mask_blend, mask_name="urban_mask_blend")
urban_polygon_blend[0]


In [ ]:
ds_blend_crop = URBAN.crop_area_city(ds = ds_crp_blend, res = domain_resolution)
da_anom_crp = ds_blend_crop['anom_tasmin'].mean(dim='time')

In [ ]:
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

fig = plt.figure(figsize=(18, 10))

gs = fig.add_gridspec(
    2, 4,
    width_ratios=[1, 1, 1, 1.5],
    height_ratios=[1, 1],
    wspace=0.08,
    hspace=0.08
)

# First 3 plots
axes = [
    fig.add_subplot(gs[0, 0], projection=plot_proj),
    fig.add_subplot(gs[0, 1], projection=plot_proj),
    fig.add_subplot(gs[0, 2], projection=plot_proj),

    # large plot
    fig.add_subplot(gs[0, 3], projection=plot_proj),

    # bottom row
    fig.add_subplot(gs[1, 0], projection=plot_proj),
    fig.add_subplot(gs[1, 1], projection=plot_proj),
    fig.add_subplot(gs[1, 2], projection=plot_proj),
]
# Dedicated space underneath plot 3
legend_ax = fig.add_subplot(gs[1, 3])
legend_ax.axis("off")

vmaxxx = 1.8
axes[0] = UHI.plot_UI_urclim_visualisation(proj = plot_proj, ax=axes[0], mask_name = 'p2p_UCN_uc', ucdb_city = ucdb_city,left_ticks = True, vmax = vmaxxx)
axes[1] = UHI_lcz.plot_UI_urclim_visualisation(proj = plot_proj, ax=axes[1], mask_name = 'p2p_UCN_lcz', ucdb_city = ucdb_city, vmax = vmaxxx)
axes[2] = UHI_m.plot_UI_urclim_visualisation(proj = plot_proj, ax=axes[2], mask_name = 'p2p_MS_uc', ucdb_city = ucdb_city, vmax = vmaxxx)

axes[3] = plot_blend_on_ax( plot_proj,  axes[3], da_anom, vmaxxx, rural_polygon_blend, urban_polygon_blend, x_station_blend, y_station_blend, uhi_blend_obs, ucdb_city)

# --- inset ---
axin =  axes[3].inset_axes( [0.53, 0.01, 0.45, 0.45], projection=plot_proj)  #[left, bottom, width, height]
plot_blend_on_ax(plot_proj, axin,da_anom_crp, vmaxxx, rural_polygon_blend, urban_polygon_blend,  x_station_blend, y_station_blend,   uhi_blend_obs,  ucdb_city,  top_labels=False, left_labels=False)

# Match inset limits to da_anom_crp
xmin, xmax = float(da_anom_crp.x.min()), float(da_anom_crp.x.max())
ymin, ymax = float(da_anom_crp.y.min()), float(da_anom_crp.y.max())

axin.set_xlim(xmin, xmax)
axin.set_ylim(ymin, ymax)

# Rectangle on main map
axes[3].add_patch( mpatches.Rectangle((xmin, ymin), xmax - xmin,    ymax - ymin, transform=plot_proj,fill=False, edgecolor='grey'))
axes[3].indicate_inset_zoom(axin, edgecolor="black")


axes[4] = UHI.plot_UI_urclim_visualisation(proj = plot_proj, ax=axes[4], mask_name = 'urmask', ucdb_city = ucdb_city ,left_ticks = True, bottom_ticks = True, vmax = vmaxxx)
axes[5] = UHI_lcz.plot_UI_urclim_visualisation(proj = plot_proj, ax=axes[5], mask_name = 'urmask', ucdb_city = ucdb_city, bottom_ticks = True, vmax = vmaxxx)
axes[6] = UHI_m.plot_UI_urclim_visualisation(proj = plot_proj, ax=axes[6], mask_name = 'urmask', ucdb_city = ucdb_city, bottom_ticks = True, vmax = vmaxxx)

# --- titles: all 15 pt ---
axes[0].set_title('p2p_UCN_uc', fontsize=15)
axes[1].set_title('p2p_UCN_lcz', fontsize=15)
axes[2].set_title('p2p_MS_uc', fontsize=15)
axes[3].set_title('p2a_MS_dist', fontsize=15)
axes[4].set_title('a2a_UCN_uc', fontsize=15)
axes[5].set_title('a2a_UCN_lcz', fontsize=15)
axes[6].set_title('a2a_MS_uc', fontsize=15)

# ============================================================
# Legend area
# ============================================================

legend_ax.axis("off")

# ---- colorbar ----
cbar_ax = legend_ax.inset_axes([0.2, 0.20, 0.7, 0.08])

sm = ScalarMappable(
    norm=Normalize(vmin=-vmaxxx, vmax=vmaxxx),
    cmap='bwr'
)

cbar = fig.colorbar(
    sm,
    extend='both',
    cax=cbar_ax,
    orientation='horizontal'
)

cbar.set_label('UHIN anomaly (°C)', fontsize=15)
cbar_ax.tick_params(labelsize=12)


# ---- symbols ----
handles = [

    Line2D(
        [0], [0],
        color='#A52A2A',
        linewidth=3,
        label='Urban simulation polygon'
    ),

    Line2D(
        [0], [0],
        color='#8A8D28',
        linewidth=3,
        label='Rural simulation polygon'
    ),

    Line2D(
        [0], [0],
        color='#ff66ff',
        linewidth=3,
        label='UCDB city boundary'
    ),

    Line2D(
        [0], [0],
        marker='o',
        linestyle='None',
        markerfacecolor='none',
        markeredgecolor='#A52A2A',
        markeredgewidth=2,
        markersize=8,
        label='Urban station'
    ),

    Line2D(
        [0], [0],
        marker='o',
        linestyle='None',
        markerfacecolor='none',
        markeredgecolor='#8A8D28',
        markeredgewidth=2,
        markersize=8,
        label='Rural station'
    ),
]

legend_ax.legend(
    handles=handles,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.82),
    frameon=False,
    ncol=1,
    fontsize=15,
    handlelength=2.5,
    columnspacing=1.5,
)
# Align the top of the large upper-right map with the other top-row maps
pos0 = axes[0].get_position()
pos3 = axes[3].get_position()

axes[3].set_position([
    pos3.x0,
    pos3.y0 - (pos3.y1 - pos0.y1),
    pos3.width,
    pos3.height
])
plt.tight_layout()
plt.savefig(f"{plot_dir}/illustration_different_methods{city}_{var}.pdf", bbox_inches='tight')
plt.show()

In [ ]:
print('figure saved to:', f"{plot_dir}/illustration_different_methods{city}_{var}.pdf")

In [ ]:
import matplotlib.patches as mpatches

fig, ax = plt.subplots(
    figsize=(10, 10),
    subplot_kw={'projection': plot_proj}
)

plot_blend_on_ax( plot_proj, ax, da_anom, vmaxxx, rural_polygon_blend, urban_polygon_blend, x_station_blend, y_station_blend, uhi_blend_obs, ucdb_city)

# --- inset ---
axin = ax.inset_axes( [0.53, 0.01, 0.45, 0.45], projection=plot_proj)  #[left, bottom, width, height]

plot_blend_on_ax(plot_proj, axin,da_anom_crp, vmaxxx, rural_polygon_blend, urban_polygon_blend,  x_station_blend, y_station_blend,   uhi_blend_obs,  ucdb_city)

# Match inset limits to da_anom_crp
xmin, xmax = float(da_anom_crp.x.min()), float(da_anom_crp.x.max())
ymin, ymax = float(da_anom_crp.y.min()), float(da_anom_crp.y.max())

axin.set_xlim(xmin, xmax)
axin.set_ylim(ymin, ymax)

# Rectangle on main map
ax.add_patch( mpatches.Rectangle((xmin, ymin), xmax - xmin,    ymax - ymin, transform=plot_proj,fill=False, edgecolor='grey'))
ax.indicate_inset_zoom(axin, edgecolor="black")

plt.show()

## Revised Fig. 1 (co-author feedback)
Same figure as above, read as a table: column headers = observational network (UCN / MS, spanning) and station classification (uc / lcz / dist), row labels = simulation aggregation (p2p / a2a / p2a). No panel titles or underscores, tilted longitude labels, UCDB boundary as a thin solid black line (instead of pink), legend and colourbar below the p2a panel with their bottom aligned with the a2a longitude labels (the p2a map is scaled to make this fit).

Three versions are saved:
1. `..._v2.pdf`: full simulated background in every panel
2. `..._v2_stationcells.pdf`: p2p panels only show the simulated UHIN in the grid cells containing a station
3. `..._v2_stationcells_maskcells.pdf`: as 2, and the a2a panels only show the simulated UHIN inside the urban + rural mask

Run after the cells above (it needs `da_anom_crp`, defined in the `crop_area_city` cell).

In [ ]:
# --- Helpers for the revised Fig. 1 (co-author feedback) --------------------------
# They post-process the axes returned by plot_UI_urclim_visualisation / plot_blend_on_ax,
# so the urclimask package itself does not need to change.
import numpy as np
import matplotlib.colors as mcolors
from matplotlib.collections import QuadMesh, PathCollection
from matplotlib.lines import Line2D
from shapely.ops import unary_union

UCDB_OLD_COLOR = '#ff66ff'   # pink used for the UCDB boundary inside urclimask
UCDB_NEW_COLOR = 'black'     # more distinct from the dark red urban simulation polygon
UCDB_NEW_LS = '-'            # solid line ...
UCDB_NEW_LW = 1.0            # ... but thinner than the simulation polygons (linewidth 2)
LON_LABEL_ROTATION = 60      # tilt of the longitude labels at the bottom (90 = vertical)


def _gridliners(ax):
    from cartopy.mpl.gridliner import Gridliner
    gls = [a for a in ax.get_children() if isinstance(a, Gridliner)]   # cartopy >= 0.23
    gls += list(getattr(ax, '_gridliners', []))                         # older cartopy
    return list({id(gl): gl for gl in gls}.values())


def tilt_lon_labels(ax, rotation=LON_LABEL_ROTATION):
    """Tilt the longitude labels so they take less horizontal space (latitude labels unchanged).
    Cartopy applies xlabel_style after its own label rotation, so this overrides it."""
    for gl in _gridliners(ax):
        gl.xlabel_style.update(rotation=rotation, rotation_mode='anchor', ha='right', va='center')


def _same_color(c, ref, tol=0.02):
    try:
        c = np.atleast_2d(mcolors.to_rgba_array(c))
    except (ValueError, TypeError):
        return False
    if c.size == 0:
        return False
    return np.all(np.abs(c[:, :3] - np.array(mcolors.to_rgb(ref))) < tol)


def recolor_artists(ax, old=UCDB_OLD_COLOR, new=UCDB_NEW_COLOR, ls=UCDB_NEW_LS, lw=UCDB_NEW_LW):
    """Restyle every line/edge drawn in colour `old` (here: the UCDB boundary)."""
    n = 0
    for art in ax.get_children():
        if isinstance(art, Line2D) and _same_color(art.get_color(), old):
            art.set_color(new); art.set_linestyle(ls); art.set_linewidth(lw); n += 1
        elif hasattr(art, 'get_edgecolor') and not isinstance(art, QuadMesh):
            if _same_color(art.get_edgecolor(), old):
                art.set_edgecolor(new); art.set_linestyle(ls); art.set_linewidth(lw); n += 1
        # cartopy FeatureArtist (add_geometries) keeps its style in _kwargs
        kw = getattr(art, '_kwargs', None)
        if isinstance(kw, dict):
            for key in ('edgecolor', 'color', 'ec'):
                if key in kw and _same_color(kw[key], old):
                    kw[key] = new; kw['linestyle'] = ls; kw['linewidth'] = lw; n += 1
    return n


def _mask_mesh(ax, keep_fn):
    """Mask every pcolormesh cell on `ax` whose centre is not selected by keep_fn(centres)."""
    for mesh in [c for c in ax.collections if isinstance(c, QuadMesh)]:
        corners = mesh.get_coordinates()                     # (ny+1, nx+1, 2)
        centres = 0.25 * (corners[:-1, :-1] + corners[1:, :-1] + corners[:-1, 1:] + corners[1:, 1:])
        arr = np.ma.asarray(mesh.get_array())
        ny, nx = centres.shape[:2]
        arr2d = arr.reshape(ny, nx)
        keep = keep_fn(centres.reshape(-1, 2)).reshape(ny, nx)
        new = np.ma.masked_array(arr2d, mask=np.ma.getmaskarray(arr2d) | ~keep)
        mesh.set_array(new if arr.ndim == 2 else new.ravel())


def keep_station_cells_only(ax):
    """p2p: only keep the grid cells that contain a station.

    Station positions are taken from the scatter(s) on the axes, grid cells from the
    pcolormesh, so this works for any panel without access to the station data."""
    pts = [c.get_offsets() for c in ax.collections
           if isinstance(c, PathCollection) and len(c.get_offsets()) > 0]
    if not pts:
        return
    pts = np.vstack(pts)

    def keep_fn(flat):
        keep = np.zeros(len(flat), dtype=bool)
        for x, y in pts:
            keep[np.argmin((flat[:, 0] - x) ** 2 + (flat[:, 1] - y) ** 2)] = True
        return keep

    _mask_mesh(ax, keep_fn)


def keep_mask_cells_only(ax, urban_vicinity, mask_name='urmask'):
    """a2a: only keep the grid cells inside the urban or rural simulation mask."""
    urban_polygon, rural_polygon = build_polygons_from_urmask(urban_vicinity, mask_name=mask_name)
    region = unary_union([urban_polygon, rural_polygon])

    def keep_fn(flat):
        try:
            from shapely import contains_xy                     # shapely >= 2.0
            return np.asarray(contains_xy(region, flat[:, 0], flat[:, 1]))
        except ImportError:
            from shapely.geometry import Point
            from shapely.prepared import prep
            pr = prep(region)
            return np.array([pr.contains(Point(x, y)) for x, y in flat])

    _mask_mesh(ax, keep_fn)


def tidy_panel(ax, stations_only=False, urban_vicinity=None, mask_name='urmask'):
    tilt_lon_labels(ax)
    recolor_artists(ax)
    if stations_only:
        keep_station_cells_only(ax)
    if urban_vicinity is not None:
        keep_mask_cells_only(ax, urban_vicinity, mask_name=mask_name)
    return ax

In [ ]:
# --- Revised Fig. 1: different evaluation methods for one city -----------------------
# Changes w.r.t. the original figure cell above (co-author feedback):
#   * layout read as a table: column headers = network (top level, spanning) and station
#     classification (second level), row labels = simulation aggregation (p2p / a2a / p2a);
#     all headers in the same (non-bold) style, no underscores
#   * no panel titles, so the a2a row sits closer to the p2p row
#   * longitude labels tilted so they take less horizontal space
#   * UCDB boundary in thin solid black instead of pink (was too close to the urban polygon colour)
#   * legend + colourbar below the p2a panel, bottom aligned with the a2a longitude labels
#   * optional: p2p panels only show the simulated UHIN in grid cells containing a station
#   * optional: a2a panels only show the simulated UHIN inside the urban + rural mask
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize

HEADER_FS = 18      # network headers and aggregation row labels
SUBHEADER_FS = 18   # classification headers
ROW_GAP = 0.02      # vertical gap between the p2p and a2a maps (figure fraction)
P2A_WIDTH = 1.5     # width of the p2a column relative to the small panels
LEGEND_FS = 14      # legend + colourbar font size (the p2a map is scaled to make the legend fit)
LEGEND_PAD = 0.01   # gap between the p2a longitude labels and the legend
CBAR_GAP, CBAR_H, CBAR_W = 0.015, 0.022, 0.9   # gap legend -> colourbar, height, width (fraction of p2a)


def _fig_bbox(fig, artist):
    """Bounding box (incl. tick/gridline labels) of an artist in figure fractions."""
    return artist.get_tightbbox(fig.canvas.get_renderer()).transformed(fig.transFigure.inverted())


def _col_header(fig, axs, network, classes, top, pad_sub=0.006, pad_net=0.05):
    """Two-level column header above `axs`: `network` spanning all of them (with a rule
    underneath), and one classification label per axes."""
    boxes = [a.get_position() for a in axs]
    for b, cls in zip(boxes, classes):
        fig.text((b.x0 + b.x1) / 2, top + pad_sub, cls, ha='center', va='bottom', fontsize=SUBHEADER_FS)
    x0, x1 = min(b.x0 for b in boxes), max(b.x1 for b in boxes)
    fig.text((x0 + x1) / 2, top + pad_net, network, ha='center', va='bottom', fontsize=HEADER_FS)
    fig.add_artist(Line2D([x0 + 0.005, x1 - 0.005], [top + pad_net - 0.005] * 2,
                          transform=fig.transFigure, color='black', linewidth=1))


def _row_label(fig, ax, label, pad=0.012):
    """Row label left of `ax`, outside its latitude labels."""
    b, tb = ax.get_position(), _fig_bbox(fig, ax)
    fig.text(min(b.x0, tb.x0) - pad, (b.y0 + b.y1) / 2, label, rotation=90, ha='right', va='center',
             fontsize=HEADER_FS)


def make_methods_figure(stations_only=False, a2a_mask_only=False, vmaxxx=1.8, savepath=None):
    fig = plt.figure(figsize=(18, 10))
    # 5th column is a narrow spacer, so the p2a row label fits next to the large p2a panel
    gs = fig.add_gridspec(2, 5, width_ratios=[1, 1, 1, 0.15, P2A_WIDTH], height_ratios=[1, 1],
                          wspace=0.08, hspace=0.05)

    axes = [
        fig.add_subplot(gs[0, 0], projection=plot_proj),
        fig.add_subplot(gs[0, 1], projection=plot_proj),
        fig.add_subplot(gs[0, 2], projection=plot_proj),
        fig.add_subplot(gs[0, 4], projection=plot_proj),   # large p2a panel
        fig.add_subplot(gs[1, 0], projection=plot_proj),
        fig.add_subplot(gs[1, 1], projection=plot_proj),
        fig.add_subplot(gs[1, 2], projection=plot_proj),
    ]

    uhi_objs = [UHI, UHI_lcz, UHI_m]   # columns: UCN-uc, UCN-lcz, MS-uc

    # p2p (top row): latitude labels on the first column only
    for i, (uhi, mask_name) in enumerate(zip(uhi_objs, ['p2p_UCN_uc', 'p2p_UCN_lcz', 'p2p_MS_uc'])):
        axes[i] = uhi.plot_UI_urclim_visualisation(proj=plot_proj, ax=axes[i], mask_name=mask_name,
                                                   ucdb_city=ucdb_city, left_ticks=(i == 0), vmax=vmaxxx)

    # p2a (top right) + inset
    axes[3] = plot_blend_on_ax(plot_proj, axes[3], da_anom, vmaxxx, rural_polygon_blend, urban_polygon_blend,
                               x_station_blend, y_station_blend, uhi_blend_obs, ucdb_city)
    axin = axes[3].inset_axes([0.53, 0.01, 0.45, 0.45], projection=plot_proj)  # [left, bottom, width, height]
    plot_blend_on_ax(plot_proj, axin, da_anom_crp, vmaxxx, rural_polygon_blend, urban_polygon_blend,
                     x_station_blend, y_station_blend, uhi_blend_obs, ucdb_city, top_labels=False, left_labels=False)
    xmin, xmax = float(da_anom_crp.x.min()), float(da_anom_crp.x.max())
    ymin, ymax = float(da_anom_crp.y.min()), float(da_anom_crp.y.max())
    axin.set_xlim(xmin, xmax)
    axin.set_ylim(ymin, ymax)
    axes[3].add_patch(mpatches.Rectangle((xmin, ymin), xmax - xmin, ymax - ymin, transform=plot_proj,
                                         fill=False, edgecolor='grey'))
    axes[3].indicate_inset_zoom(axin, edgecolor="black")

    # a2a (bottom row): longitude labels on every panel, latitude labels on the first column
    for i, uhi in zip((4, 5, 6), uhi_objs):
        axes[i] = uhi.plot_UI_urclim_visualisation(proj=plot_proj, ax=axes[i], mask_name='urmask',
                                                   ucdb_city=ucdb_city, left_ticks=(i == 4), bottom_ticks=True,
                                                   vmax=vmaxxx)

    # tilted lon labels, UCDB restyled, optionally station cells only (p2p) / mask cells only (a2a)
    for i, ax in enumerate(axes):
        tidy_panel(ax,
                   stations_only=(stations_only and i in (0, 1, 2)),
                   urban_vicinity=(uhi_objs[i - 4].urban_vicinity if a2a_mask_only and i in (4, 5, 6) else None))
    tidy_panel(axin)

    # top-align every map within its grid cell (also the larger p2a map) ...
    for ax in axes:
        ax.set_anchor('N')
    fig.canvas.draw()
    # ... and pull the a2a row up to just below the p2p row (there are no panel titles any more)
    p2p_bottom = min(axes[i].get_position().y0 for i in (0, 1, 2))
    for i in (4, 5, 6):
        b = axes[i].get_position()
        axes[i].set_position([b.x0, p2p_bottom - ROW_GAP - b.height, b.width, b.height])
    fig.canvas.draw()

    # --- legend + colourbar below the p2a panel, bottom aligned with the a2a longitude labels ---
    # The legend has a fixed font size; the p2a map (with its inset) is scaled around its top
    # centre so that the legend block fits exactly between the p2a longitude labels and the
    # bottom of the a2a longitude labels.
    handles = [
        Line2D([0], [0], color='#A52A2A', linewidth=2, label='Urban simulation polygon'),
        Line2D([0], [0], color='#8A8D28', linewidth=2, label='Rural simulation polygon'),
        Line2D([0], [0], color=UCDB_NEW_COLOR, linestyle=UCDB_NEW_LS, linewidth=UCDB_NEW_LW, label='UCDB urban centre'),
        Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='none', markeredgecolor='#A52A2A',
               markeredgewidth=2, markersize=8, label='Urban station'),
        Line2D([0], [0], marker='o', linestyle='None', markerfacecolor='none', markeredgecolor='#8A8D28',
               markeredgewidth=2, markersize=8, label='Rural station'),
    ]
    sm = ScalarMappable(norm=Normalize(vmin=-vmaxxx, vmax=vmaxxx), cmap='bwr')

    def place_legend_block(leg_top):
        """Legend with its top at `leg_top`, colourbar below it (figure fractions)."""
        p2a = axes[3].get_position()
        xc, cbar_w = (p2a.x0 + p2a.x1) / 2, CBAR_W * p2a.width
        leg = fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(xc, leg_top),
                         bbox_transform=fig.transFigure, frameon=False, ncol=1, fontsize=LEGEND_FS,
                         handlelength=2.5, labelspacing=0.4)
        fig.canvas.draw()
        cbar_ax = fig.add_axes([xc - cbar_w / 2, _fig_bbox(fig, leg).y0 - CBAR_GAP - CBAR_H, cbar_w, CBAR_H])
        cbar = fig.colorbar(sm, extend='both', cax=cbar_ax, orientation='horizontal')
        cbar.set_label('UHIN anomaly (°C)', fontsize=LEGEND_FS)
        cbar_ax.tick_params(labelsize=LEGEND_FS - 2)
        fig.canvas.draw()
        return leg, cbar_ax, _fig_bbox(fig, cbar_ax).y0

    a2a_bottom = min(_fig_bbox(fig, axes[i]).y0 for i in (4, 5, 6))     # below the a2a longitude labels

    # 1) measure the height of the legend block
    leg, cbar_ax, block_bottom = place_legend_block(0.5)
    block_h = 0.5 - block_bottom
    leg.remove(); cbar_ax.remove()

    # 2) scale the p2a map so that labels + gap + legend block end at the a2a bottom
    pos = axes[3].get_position()
    labels_h = pos.y0 - _fig_bbox(fig, axes[3]).y0                      # p2a longitude labels
    new_h = pos.y1 - (a2a_bottom + block_h + LEGEND_PAD + labels_h)
    scale = min(new_h / pos.height, 1.0)                                  # never larger than its grid cell
    xc = (pos.x0 + pos.x1) / 2
    axes[3].set_position([xc - scale * pos.width / 2, pos.y1 - scale * pos.height,
                          scale * pos.width, scale * pos.height])
    fig.canvas.draw()

    # 3) place the block below the p2a labels and move it so its bottom is at the a2a bottom
    leg, cbar_ax, block_bottom = place_legend_block(_fig_bbox(fig, axes[3]).y0 - LEGEND_PAD)
    dy = a2a_bottom - block_bottom
    leg.set_bbox_to_anchor((xc, _fig_bbox(fig, axes[3]).y0 - LEGEND_PAD + dy), transform=fig.transFigure)
    b = cbar_ax.get_position()
    cbar_ax.set_position([b.x0, b.y0 + dy, b.width, b.height])
    fig.canvas.draw()

    # --- table-like labelling ---
    top = max(axes[i].get_position().y1 for i in range(4))
    _col_header(fig, axes[0:2], 'UCN', ['uc', 'lcz'], top)
    _col_header(fig, axes[2:3], 'MS', ['uc'], top)
    _col_header(fig, axes[3:4], 'MS', ['dist'], top)
    _row_label(fig, axes[0], 'p2p')
    _row_label(fig, axes[4], 'a2a')
    _row_label(fig, axes[3], 'p2a')

    if savepath is not None:
        fig.savefig(savepath, bbox_inches='tight')
        print('figure saved to:', savepath)
    return fig


fig_base = f"{plot_dir}/illustration_different_methods{city}_{var}_v2"

# version 1: full simulated background map in every panel
make_methods_figure(savepath=f"{fig_base}.pdf")
plt.show()

# version 2: p2p panels only show the simulated UHIN in the station grid cells
make_methods_figure(stations_only=True, savepath=f"{fig_base}_stationcells.pdf")
plt.show()

# version 3: as version 2, and a2a panels only show the simulated UHIN inside the urban + rural mask
make_methods_figure(stations_only=True, a2a_mask_only=True, savepath=f"{fig_base}_stationcells_maskcells.pdf")
plt.show()

In [ ]:
np.array(method_spread_sim) / uhis_df.filter(like='sim').abs().mean(axis=1).values